# Step 4: Demand Forecasting

**Objective:** Predict sales for popular categories.

**File:** `notebooks/4_forecasting.ipynb`

## Actions:
1. Aggregate data weekly by product category.
2. Train Prophet Model for top categories.
3. Visualize historical + prediction.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
import os

# Mute warnings
import logging
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

In [ ]:
# Load cleaned data
input_file = '../dataset/cleaned_data.pkl'
df = pd.read_pickle(input_file)

In [ ]:
# Calculate Weekly Sales
def calculate_weekly_sales(df, category):
    category_df = df[df['product_category_name_english'] == category].copy()
    category_df['order_purchase_timestamp'] = pd.to_datetime(category_df['order_purchase_timestamp'])
    weekly_sales = category_df.groupby(pd.Grouper(key='order_purchase_timestamp', freq='W'))['price'].sum().reset_index()
    return weekly_sales

In [ ]:
# Run Forecasting for Top Categories
top_categories = df['product_category_name_english'].value_counts().head(3).index.tolist()

for category in top_categories:
    print(f"Forecasting for {category}...")
    weekly_sales = calculate_weekly_sales(df, category)
    
    if len(weekly_sales) > 10:
        # Prepare for Prophet
        prophet_df = weekly_sales.rename(columns={'order_purchase_timestamp': 'ds', 'price': 'y'})
        
        # Model
        model = Prophet(yearly_seasonality=True, weekly_seasonality=True)
        model.fit(prophet_df)
        
        # Forecast
        future = model.make_future_dataframe(periods=12, freq='W')
        forecast = model.predict(future)
        
        # Plot
        fig1 = model.plot(forecast)
        plt.title(f'Sales Forecast for {category}')
        plt.show()
        
        fig2 = model.plot_components(forecast)
        plt.show()
    else:
        print(f"Not enough data for {category}")